# Урок 21. Базы данных: таблицы, поля, ключи

9 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [← Урок 20](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-20.ipynb) · [Урок 22 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-22.ipynb)

---

Зачем нужны базы данных. Реляционная модель: таблица, запись, поле, тип поля. Первичный ключ и связи между таблицами. Практика в Access: три таблицы, типы полей, целостность.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 9А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
for адрес in ("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py",
              "https://cdn.jsdelivr.net/gh/LeksssGray/informatika-7-11@main/lib/schoolinf.py"):
    try:
        urllib.request.urlretrieve(адрес, "schoolinf.py")
        break
    except Exception as ошибка:
        print("⚠️ Не скачалась библиотека проверки:", ошибка)
else:
    raise SystemExit("Проверь интернет и запусти эту ячейку ещё раз. "
                     "Не помогло — покажи это сообщение учителю.")
import schoolinf as si
importlib.reload(si)
si.start(lesson="09-21", name=ФИО, klass=Класс)

## Разбираемся

### Таблица, которая перестала помогать

Школьная библиотека ведёт учёт в одной таблице:

| Книга | Автор | Год | Читатель | Класс | Телефон | Выдана |
|---|---|---|---|---|---|---|
| Мастер и Маргарита | Булгаков | 1967 | Иванов Иван | 9А | 555-11-22 | 01.09 |
| Война и мир | Толстой | 1869 | Иванов Иван | 9А | 555-11-22 | 05.09 |
| Мастер и Маргарита | Булгаков | 1967 | Петрова Аня | 9Б | 555-33-44 | 12.09 |

Всё видно, всё понятно — и всё плохо:

* **данные дублируются**: телефон Иванова записан дважды, название
  книги — тоже;
* **правка разъезжается**: сменился телефон — исправили в одной
  строке, забыли в другой, и теперь у одного человека два телефона;
* **опечатки не видно**: «Булгаков» и «Булгаков М.» для таблицы —
  два разных автора;
* **нельзя записать то, чего ещё нет**: книгу, которую никто
  не брал, добавить некуда — строка про выдачу пустая.

Лечится это одним приёмом: **одна сущность — одна таблица**,
а связи между ними — через ключи. Такой способ хранения называется
**реляционной моделью**, и на ней работают почти все базы данных мира.

### Поле, запись, тип

| Термин | Что это |
|---|---|
| **Поле** | столбец: один признак у всех объектов |
| **Запись** | строка: один объект целиком |
| **Тип поля** | что в нём разрешено хранить |

Тип поля — не формальность. Он защищает данные: в поле типа
«целое» не влезет «пять», а в поле «дата» — «вчера».

| Тип | Что хранит | Пример |
|---|---|---|
| целое (`INTEGER`) | числа без дробной части | 1967 |
| вещественное (`REAL`) | числа с дробью | 4.75 |
| текст (`TEXT`) | строки | «Булгаков» |
| дата (`DATE`) | даты | 2026-09-01 |
| логический (`BOOLEAN`) | да или нет | истина |

### Первичный ключ

> **Первичный ключ** — поле или набор полей, значение которых
> не повторяется ни у одной записи и никогда не бывает пустым.

По нему запись находят однозначно. Фамилия ключом быть не может:
Ивановых двое. Название книги — тоже: одна книга может быть
в библиотеке в пяти экземплярах.

Если естественного ключа нет, его **заводят**: столбец `id`
с числами 1, 2, 3… Так устроено большинство реальных баз,
и так проще: номер никогда не изменится, а фамилия — может.

### Связи и внешний ключ

> **Внешний ключ** — поле, которое хранит значение первичного ключа
> другой таблицы.

Именно он и связывает таблицы. Разложим библиотеку правильно:

```
Книги                         Читатели
┌────┬──────────────┬────────┐ ┌────┬────────────┬───────┬───────────┐
│ id │ название     │ автор  │ │ id │ фамилия    │ класс │ телефон   │
├────┼──────────────┼────────┤ ├────┼────────────┼───────┼───────────┤
│ 1  │ Мастер и М.  │ Булг.  │ │ 1  │ Иванов     │ 9А    │ 555-11-22 │
│ 2  │ Война и мир  │ Толст. │ │ 2  │ Петрова    │ 9Б    │ 555-33-44 │
└────┴──────────────┴────────┘ └────┴────────────┴───────┴───────────┘

                    Выдачи
          ┌────┬───────┬──────────┬────────────┐
          │ id │ книга │ читатель │ дата       │
          ├────┼───────┼──────────┼────────────┤
          │ 1  │   1   │    1     │ 2026-09-01 │
          │ 2  │   2   │    1     │ 2026-09-05 │
          │ 3  │   1   │    2     │ 2026-09-12 │
          └────┴───────┴──────────┴────────────┘
```

Теперь телефон Иванова хранится **в одном месте**. Поменялся —
исправили одну ячейку, и во всех отчётах он новый.

### Какие бывают связи

| Связь | Что означает | Пример |
|---|---|---|
| **один к одному** | одной записи соответствует ровно одна | ученик ↔ его читательский билет |
| **один ко многим** | одной записи — много записей второй таблицы | один читатель — много выдач |
| **многие ко многим** | и там, и там много | книги ↔ читатели: одну книгу брали многие, один читатель брал многие |

Связь «многие ко многим» напрямую не делают — для неё заводят
**третью таблицу**, как наши «Выдачи». Она и хранит пары
«кто что взял».

### Разложить данные правильно

Процесс раскладывания по таблицам без повторов называется
**нормализацией**. Для школьного уровня достаточно трёх вопросов
к любой таблице:

1. Хранится ли что-то дважды? Значит, это отдельная таблица.
2. Есть ли у каждой записи ключ, который точно не повторится?
3. Если исправить один факт — придётся ли править несколько ячеек?
   Если да, разложено неверно.

Проверка номер три — самая практичная. Если смена телефона требует
двадцати правок, база спроектирована плохо, и рано или поздно
где-нибудь останется старый номер.

## Практическая работа. Библиотека из трёх таблиц

Работаем в **Microsoft Access**. Разложим библиотеку правильно:
«Книги», «Читатели» и «Выдачи» — третья таблица, которая хранит
связь «многие ко многим».

[Готовая база с урока](https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/files/access/g09-21-biblioteka.accdb) — если
не успеете или захотите сверить свою.

Создайте новую базу: **Файл → Создать → Пустая база данных**, имя
`Библиотека`. Если появится полоса **Предупреждение системы
безопасности** — **Включить содержимое**. Пустую «Таблицу1»
закройте.

### Шаг 1. «Книги»: пять разных типов

**Создание → Конструктор таблиц**:

| Имя поля | Тип данных | Что даёт тип |
|---|---|---|
| Код | Счетчик, **ключевое поле** | номер проставится сам и не повторится |
| Название | Короткий текст | до 255 символов |
| Автор | Короткий текст | |
| Год | Числовой | «тысяча девятьсот» сюда не впишешь |
| Цена | Денежный | два знака после запятой и знак ₽ |
| ВНаличии | Логический | галочка: да или нет |

Внизу, в **Свойствах поля**, у каждого типа свои настройки.
У числового поля «Год» в строке **Размер поля** — «Длинное целое»;
у текстовых можно уменьшить длину с 255 до 40 — для фамилии хватит.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-21-tipy.png" width="900" alt="Конструктор таблицы «Книги»">

*Конструктор таблицы «Книги»*

Сохраните как `Книги` и заполните восемь книг:

| Название | Автор | Год | Цена | ВНаличии |
|---|---|---|---|---|
| Мастер и Маргарита | Булгаков | 1967 | 540 | ✓ |
| Война и мир | Толстой | 1869 | 890 | ✓ |
| Капитанская дочка | Пушкин | 1836 | 250 | ✓ |
| Мёртвые души | Гоголь | 1842 | 380 | |
| Преступление и наказание | Достоевский | 1866 | 610 | ✓ |
| Отцы и дети | Тургенев | 1862 | 320 | ✓ |
| Горе от ума | Грибоедов | 1825 | 210 | |
| Собачье сердце | Булгаков | 1925 | 290 | ✓ |

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-21-knigi.png" width="900" alt="Таблица «Книги»">

*Таблица «Книги»*

### Шаг 2. Тип защищает данные

Встаньте в пустую строку внизу и впишите книгу «Бедная Лиза»,
автор «Карамзин», а в «Год» — слово `пять`. Нажмите `Tab`.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-21-oshibka-tipa.png" width="900" alt="Не тот тип данных">

*Не тот тип данных*

Access не пустит: в числовое поле текст не записывается. Нажмите
`Esc`, чтобы отменить строку. Именно так тип поля защищает базу
от мусора — ещё до того, как он попадёт в отчёты.

### Шаг 3. «Читатели»

Конструктор таблиц:

| Имя поля | Тип данных |
|---|---|
| Код | Счетчик, ключевое поле |
| Фамилия | Короткий текст |
| Имя | Короткий текст |
| Класс | Короткий текст |
| Телефон | Короткий текст |

Телефон — **текст**, хотя состоит из цифр: считать с ним никто
не будет, а дефисы и ведущий ноль числовое поле съест.

| Фамилия | Имя | Класс | Телефон |
|---|---|---|---|
| Иванов | Иван | 9А | 555-11-22 |
| Петрова | Анна | 9Б | 555-33-44 |
| Сидоров | Максим | 9А | 555-12-80 |
| Иванов | Кирилл | 9В | 555-40-17 |
| Кузнецова | Мария | 9Б | 555-71-03 |
| Орлов | Денис | 9В | 555-58-26 |
| Никитина | Ольга | 9А | 555-90-44 |
| Жуков | Илья | 9Б | 555-26-61 |

Заметьте: Ивановых двое. Фамилия ключом быть не может — а код может.

### Шаг 4. «Выдачи» и схема данных

Конструктор таблиц:

| Имя поля | Тип данных |
|---|---|
| Код | Счетчик, ключевое поле |
| Книга | Числовой — код книги |
| Читатель | Числовой — код читателя |
| Дата | Дата и время |

Сохраните и **не заполняйте**. Откройте **Работа с базами данных →
Схема данных**, добавьте все три таблицы и протяните две связи:

* «Код» из «Книги» → на «Книга» в «Выдачи»;
* «Код» из «Читатели» → на «Читатель» в «Выдачи».

В окне **Изменение связей** каждый раз ставьте галочку
**Обеспечение целостности данных** → **Создать**.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-21-shema.png" width="900" alt="Схема: две связи «один ко многим»">

*Схема: две связи «один ко многим»*

У каждой линии — **1** со стороны справочника и **∞** со стороны
«Выдач». Две связи «один ко многим» вместе и дают «многие ко
многим»: одну книгу берут многие, один читатель берёт многие книги.

### Шаг 5. Выдачи и целостность

Заполните «Выдачи» (коды книг и читателей — из первых двух таблиц):

| Книга | Читатель | Дата | | Книга | Читатель | Дата |
|---|---|---|---|---|---|---|
| 1 | 1 | 01.09.2026 | | 3 | 6 | 09.09.2026 |
| 2 | 1 | 05.09.2026 | | 1 | 7 | 10.09.2026 |
| 1 | 2 | 12.09.2026 | | 4 | 2 | 17.09.2026 |
| 3 | 3 | 03.09.2026 | | 8 | 8 | 18.09.2026 |
| 5 | 4 | 04.09.2026 | | 3 | 1 | 22.09.2026 |
| 6 | 5 | 08.09.2026 | | 2 | 5 | 23.09.2026 |
| 8 | 1 | 15.09.2026 | | 6 | 3 | 24.09.2026 |

Теперь попробуйте выдать книгу читателю `99`:

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-21-oshibka-celostnosti.png" width="900" alt="Нарушение целостности">

*Нарушение целостности*

Читателя с кодом 99 нет — Access не даст записать выдачу «в никуда».
**ОК**, затем `Esc`.

### Шаг 6. Подтаблицы

Откройте «Читатели». Слева у каждой записи появился **плюсик** —
это связь в действии. Нажмите его у Ивана Иванова: раскроются
все его выдачи.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-21-podtablica.png" width="900" alt="Подтаблица выдач читателя">

*Подтаблица выдач читателя*

### Шаг 7. Собираем обратно запросом

В «Выдачах» — только номера. Чтобы увидеть «кто, что, когда»,
соберём три таблицы запросом: **Создание → Конструктор запросов**,
добавьте все три таблицы. Линии связей Access нарисует сам.
В бланк: «Фамилия» и «Класс» из «Читатели», «Название» из «Книги»,
«Дата» из «Выдачи»; сортировка по дате.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-21-tri-tablicy.png" width="900" alt="Запрос по трём таблицам">

*Запрос по трём таблицам*

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-21-tri-tablicy-rezultat.png" width="900" alt="Кто что взял">

*Кто что взял*

Сохраните как `Кто что взял`.

## Проверяем себя

В задачах таблицы приходят списками записей, как в примерах:
`[id, фамилия, класс]` и `[id, книга, читатель]`.

### Задача 1. Как называется ключ

Как называется поле, значение которого не повторяется ни у одной
записи и однозначно определяет её?

In [ ]:
#@title 🧩 Задача 1. Название ключа { display-mode: "form" }
#@markdown Выбери термин
ключ = "выбери ответ" #@param ["выбери ответ", "первичный ключ", "внешний ключ", "индекс", "тип поля"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("1", ключ, "1ac0b46e443db517",
         hint="Тот, что объявляется словами PRIMARY KEY.")

### Задача 2. Поле, которое ссылается

Как называется поле, которое хранит значение ключа другой таблицы
и связывает таблицы между собой?

In [ ]:
#@title 🧩 Задача 2. Связующее поле { display-mode: "form" }
#@markdown Выбери термин
связь_полей = "выбери ответ" #@param ["выбери ответ", "первичный ключ", "внешний ключ", "составной ключ"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("2", связь_полей, "6a32d952f0fa44a4",
         hint="FOREIGN KEY по-русски.")

### Задача 3. Тип связи

Один читатель может взять много книг, но каждая конкретная выдача
относится к одному читателю. Как называется такая связь?

In [ ]:
#@title 🧩 Задача 3. Тип связи { display-mode: "form" }
#@markdown Выбери связь
тип_связи = "выбери ответ" #@param ["выбери ответ", "один к одному", "один ко многим", "многие ко многим"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("3", тип_связи, "63f90a90bd02bbb0",
         hint="Со стороны читателя — много, со стороны выдачи — один.")

### Задача 4. Найти по ключу

Какой год издания у книги с кодом 5? Найдите её в таблице «Книги».

In [ ]:
#@title 🧩 Задача 4. Запись по ключу { display-mode: "form" }
#@markdown Впиши год
год = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("4", год, "bd54ada8575526cd",
         hint="Код 5 — «Преступление и наказание».")

### Задача 5. Подтаблица

Сколько книг брал Иван Иванов (читатель с кодом 1)? Раскройте
плюсик у его записи.

In [ ]:
#@title 🧩 Задача 5. Выдачи одного читателя { display-mode: "form" }
#@markdown Впиши число
у_иванова = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("5", у_иванова, "4b227777d4dd1fc6",
         hint="Именно у Ивана, код 1, — не у Кирилла.")

### Задача 6. Популярная книга

Сколько раз брали «Мастера и Маргариту»? Откройте «Книги»
и раскройте плюсик у неё — подтаблицы работают в обе стороны.

In [ ]:
#@title 🧩 Задача 6. Выдачи одной книги { display-mode: "form" }
#@markdown Впиши число
мастер = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("6", мастер, "4e07408562bedb8b",
         hint="Книга с кодом 1.")

## Домашнее задание

### Домашнее задание 1. База «многие ко многим»

Сделайте в Access базу из **трёх** таблиц со связью «многие ко
многим» на свою тему. Примеры пар: ученики и кружки (таблица
«Записи»), фильмы и актёры («Роли»), покупатели и товары
(«Покупки»), игроки и турниры («Участия»).

Требования:

1. две таблицы-справочника и одна связующая; все созданы
   в конструкторе, у каждой ключ-счётчик;
2. во всех таблицах вместе — **не меньше четырёх разных типов**
   данных: текст, числовой, дата, денежный, логический;
3. у текстовых полей **Размер поля** уменьшен до разумного
   (фамилия — 40, класс — 5);
4. в связующей таблице — два числовых поля-ссылки, на схеме
   данных — две связи с **обеспечением целостности**;
5. в справочниках — **по 5 записей и больше**, в связующей —
   **не меньше 12**;
6. запрос по всем трём таблицам, который показывает связи
   словами («Иванов — шахматы»), с сортировкой.

Под схемой в тетради (или в описании поля любой таблицы) ответьте:
сколько ячеек придётся исправить, если в первом справочнике
сменится название? Если больше одной — разложено неверно.

Сдайте учителю файл `.accdb`, закрыв перед этим Access.

### Домашнее задание 2. Годится ли в ключи

Сколько раз в таблице «Читатели» встречается фамилия Иванов?
Отсортируйте таблицу по фамилии (щелчок по стрелочке у заголовка
«Фамилия» → **Сортировка от А до Я**) — повторы окажутся рядом.

In [ ]:
#@title 🏠 Домашнее 2. Повторы фамилии { display-mode: "form" }
#@markdown Впиши число
иванов = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("дз2", иванов, "d4735e3a265e16ee",
         hint="Иван и Кирилл.")

### Домашнее задание 3. Сколько таблиц

Сколько таблиц нужно, чтобы связать книги и читателей связью
«многие ко многим», не считая саму связующую таблицу?

In [ ]:
#@title 🏠 Домашнее 3. Сколько таблиц { display-mode: "form" }
#@markdown Впиши число
таблиц = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("дз3", таблиц, "d4735e3a265e16ee",
         hint="Книги и читатели — это сколько таблиц?")

---

### Любопытно

Реляционную модель придумал в 1970 году математик Эдгар Кодд.
В IBM, где он работал, идею встретили прохладно: у компании уже была
своя база данных, продавать её было выгоднее, и Кодду разрешили
«заниматься теорией».

Статью прочитали в маленькой калифорнийской компании и сделали
по ней продукт. Компания называлась Oracle и стоит сегодня
сотни миллиардов долларов.

А Кодд получил премию Тьюринга — высшую награду в информатике.
За таблицы с ключами.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 20](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-20.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 22 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-22.ipynb)